In [1]:
!pip install nibabel tqdm --quiet

In [2]:
import os
import time
import random
import numpy as np
import nibabel as nib
from glob import glob
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [3]:
SEED = 42
IMG_SIZE = 128
EPOCHS = 10
BATCH_SIZE = 4
LR = 0.01
NUM_WORKERS = 4
PATIENCE = 6

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:", torch.cuda.get_device_name(i))

CUDA: True
GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [4]:
DATA_ROOT = "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation"
TRAIN_PATH = os.path.join(DATA_ROOT, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")

all_cases = sorted(glob(os.path.join(TRAIN_PATH, "BraTS20_Training_*")))
all_cases = [c for c in all_cases if os.path.exists(f"{c}/{os.path.basename(c)}_seg.nii")]

print("Total cases:", len(all_cases))

Total cases: 368


In [5]:
def load_and_preprocess(path):
    cid = os.path.basename(path)

    def load_mod(mod):
        vol = nib.load(f"{path}/{cid}_{mod}.nii").get_fdata().astype(np.float32)
        mask = vol != 0
        if mask.sum() > 0:
            vol[mask] = (vol[mask] - vol[mask].mean()) / (vol[mask].std() + 1e-8)
        t = torch.from_numpy(vol).unsqueeze(0).unsqueeze(0)
        t = F.interpolate(
            t,
            size=(IMG_SIZE, IMG_SIZE, IMG_SIZE),
            mode="trilinear",
            align_corners=False,
        )
        return t.squeeze().numpy().astype(np.float16)

    flair = load_mod("flair")
    t1 = load_mod("t1")
    t1ce = load_mod("t1ce")
    t2 = load_mod("t2")

    seg = nib.load(f"{path}/{cid}_seg.nii").get_fdata().astype(np.float32)
    t = torch.from_numpy(seg).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=(IMG_SIZE, IMG_SIZE, IMG_SIZE), mode="nearest")
    seg = t.squeeze().numpy()

    wt = (seg > 0).astype(np.float16)
    tc = ((seg == 1) | (seg == 4)).astype(np.float16)
    et = (seg == 4).astype(np.float16)

    image = np.stack([flair, t1, t1ce, t2], axis=0)
    label = np.stack([wt, tc, et], axis=0)
    return image, label


print("Preprocessing and caching all volumes to RAM...")
cache_start = time.time()

image_cache = {}
label_cache = {}

for path in tqdm(all_cases):
    img, lbl = load_and_preprocess(path)
    image_cache[path] = img
    label_cache[path] = lbl

cache_time = time.time() - cache_start
print(f"Cache built in {cache_time:.1f}s  ({cache_time/60:.1f}min)")

total_mb = sum(v.nbytes for v in image_cache.values()) + sum(v.nbytes for v in label_cache.values())
print(f"Total RAM used: {total_mb / 1e9:.2f} GB")

Preprocessing and caching all volumes to RAM...


  0%|          | 0/368 [00:00<?, ?it/s]

Cache built in 637.5s  (10.6min)
Total RAM used: 10.80 GB


In [6]:
class BraTSDataset(Dataset):
    def __init__(self, cases, augment=False):
        self.cases = cases
        self.augment = augment

    def __len__(self):
        return len(self.cases)

    def __getitem__(self, idx):
        path = self.cases[idx]
        image = torch.from_numpy(image_cache[path].astype(np.float32))
        label = torch.from_numpy(label_cache[path].astype(np.float32))

        if self.augment:
            for ax in [1, 2, 3]:
                if random.random() < 0.5:
                    image = torch.flip(image, [ax])
                    label = torch.flip(label, [ax])
            image = image * torch.empty(4, 1, 1, 1).uniform_(0.9, 1.1)
            image = image + torch.empty(4, 1, 1, 1).uniform_(-0.1, 0.1)

        return image, label

In [7]:
train_ids, temp_ids = train_test_split(all_cases, test_size=0.3, random_state=SEED)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=SEED)

print("Train:", len(train_ids), " Val:", len(val_ids), " Test:", len(test_ids))

train_loader = DataLoader(
    BraTSDataset(train_ids, augment=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True,
)
val_loader = DataLoader(
    BraTSDataset(val_ids, augment=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)
test_loader = DataLoader(
    BraTSDataset(test_ids, augment=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

print("Train batches:", len(train_loader))
print("Val batches:  ", len(val_loader))

Train: 257  Val: 55  Test: 56
Train batches: 64
Val batches:   14


In [8]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1 = DoubleConv(4, 32)
        self.e2 = DoubleConv(32, 64)
        self.e3 = DoubleConv(64, 128)
        self.b = DoubleConv(128, 256)
        self.pool = nn.MaxPool3d(2)
        self.u3 = nn.ConvTranspose3d(256, 128, 2, 2)
        self.d3 = DoubleConv(256, 128)
        self.u2 = nn.ConvTranspose3d(128, 64, 2, 2)
        self.d2 = DoubleConv(128, 64)
        self.u1 = nn.ConvTranspose3d(64, 32, 2, 2)
        self.d1 = DoubleConv(64, 32)
        self.out = nn.Conv3d(32, 3, 1)
        self.ds2 = nn.Conv3d(64, 3, 1)
        self.ds3 = nn.Conv3d(128, 3, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        b = self.b(self.pool(e3))
        d3 = self.d3(torch.cat([self.u3(b), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        return self.out(d1), self.ds2(d2), self.ds3(d3)

In [9]:
def region_loss(logits, target):
    pred = torch.sigmoid(logits)
    inter = (pred * target).sum(dim=(0, 2, 3, 4))
    union = pred.sum(dim=(0, 2, 3, 4)) + target.sum(dim=(0, 2, 3, 4))
    dice = 1 - (2 * inter + 1e-6) / (union + 1e-6)
    bce = F.binary_cross_entropy_with_logits(logits, target)
    return dice.mean() + bce


def deep_supervision_loss(outputs, target):
    main, ds2, ds3 = outputs
    t2 = F.interpolate(target, size=ds2.shape[2:], mode="nearest")
    t3 = F.interpolate(target, size=ds3.shape[2:], mode="nearest")
    return (
        1.00 * region_loss(main, target) + 0.50 * region_loss(ds2, t2) + 0.25 * region_loss(ds3, t3)
    )

In [10]:
model = UNet3D()
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(DEVICE)

optimizer = torch.optim.SGD(
    model.parameters(), lr=LR, momentum=0.99, nesterov=True, weight_decay=3e-5
)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda ep: (1 - ep / EPOCHS) ** 0.9
)
scaler = torch.amp.GradScaler("cuda")

print("Parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Parameters: 5,604,361


In [11]:
def dice_score(pred_bin, target_bin):
    inter = (pred_bin & target_bin).sum().float()
    return ((2 * inter + 1e-6) / (pred_bin.sum() + target_bin.sum() + 1e-6)).item()


def brats_metrics(pred_logits, target):
    pred = torch.sigmoid(pred_logits) > 0.5
    wt = dice_score(pred[:, 0], target[:, 0].bool())
    tc = dice_score(pred[:, 1], target[:, 1].bool())
    et = dice_score(pred[:, 2], target[:, 2].bool())
    return wt, tc, et

In [12]:
best_mean_dice = 0.0
patience_counter = 0
train_start = time.time()

for epoch in range(EPOCHS):
    t0 = time.time()
    current_lr = scheduler.get_last_lr()[0]
    elapsed = (t0 - train_start) / 60
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{EPOCHS}   LR: {current_lr:.5f}   Elapsed: {elapsed:.1f}min")

    model.train()
    train_loss = 0.0
    for x, y in tqdm(train_loader, desc="Train", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            outputs = model(x)
            loss = deep_supervision_loss(outputs, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    scheduler.step()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    wt_scores, tc_scores, et_scores = [], [], []

    with torch.no_grad():
        for x, y in tqdm(val_loader, desc="Val", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.amp.autocast("cuda"):
                outputs = model(x)
                loss = deep_supervision_loss(outputs, y)
            val_loss += loss.item()
            wt, tc, et = brats_metrics(outputs[0], y)
            wt_scores.append(wt)
            tc_scores.append(tc)
            et_scores.append(et)

    val_loss /= len(val_loader)
    wt_mean = np.mean(wt_scores)
    tc_mean = np.mean(tc_scores)
    et_mean = np.mean(et_scores)
    mean_dice = (wt_mean + tc_mean + et_mean) / 3
    epoch_time = time.time() - t0

    print(f"Train Loss: {train_loss:.4f}   Val Loss: {val_loss:.4f}")
    print(f"WT: {wt_mean:.4f}   TC: {tc_mean:.4f}   ET: {et_mean:.4f}   Mean: {mean_dice:.4f}")
    print(f"Epoch time: {epoch_time:.1f}s   Total elapsed: {(time.time()-train_start)/60:.1f}min")

    if mean_dice > best_mean_dice:
        best_mean_dice = mean_dice
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"Best model saved  (mean dice: {best_mean_dice:.4f})")
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

print(f"\nTraining complete. Total time: {(time.time()-train_start)/60:.1f}min")


Epoch 1/10   LR: 0.01000   Elapsed: 0.0min


Train:   0%|          | 0/64 [00:00<?, ?it/s]

Val:   0%|          | 0/14 [00:00<?, ?it/s]

Train Loss: 1.8430   Val Loss: 0.8222
WT: 0.7889   TC: 0.6631   ET: 0.6422   Mean: 0.6980
Epoch time: 184.7s   Total elapsed: 3.1min
Best model saved  (mean dice: 0.6980)

Epoch 2/10   LR: 0.00910   Elapsed: 3.1min


Train:   0%|          | 0/64 [00:00<?, ?it/s]

...

Val:   0%|          | 0/14 [00:00<?, ?it/s]

Train Loss: 0.3962   Val Loss: 0.4119
WT: 0.8697   TC: 0.7904   ET: 0.7397   Mean: 0.8000
Epoch time: 197.0s   Total elapsed: 29.1min
Best model saved  (mean dice: 0.8000)

Epoch 10/10   LR: 0.00126   Elapsed: 29.1min


Train:   0%|          | 0/64 [00:00<?, ?it/s]

Val:   0%|          | 0/14 [00:00<?, ?it/s]

Train Loss: 0.3852   Val Loss: 0.4061
WT: 0.8729   TC: 0.7844   ET: 0.7488   Mean: 0.8021
Epoch time: 198.4s   Total elapsed: 32.4min
Best model saved  (mean dice: 0.8021)

Training complete. Total time: 32.4min


In [13]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

test_loss = 0.0
wt_scores, tc_scores, et_scores = [], [], []

with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Test"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.amp.autocast("cuda"):
            outputs = model(x)
            loss = deep_supervision_loss(outputs, y)
        test_loss += loss.item()
        wt, tc, et = brats_metrics(outputs[0], y)
        wt_scores.append(wt)
        tc_scores.append(tc)
        et_scores.append(et)

test_loss /= len(test_loader)
wt_mean = np.mean(wt_scores)
tc_mean = np.mean(tc_scores)
et_mean = np.mean(et_scores)
mean_dice = (wt_mean + tc_mean + et_mean) / 3

print(f"\n{'='*60}")
print("FINAL TEST RESULTS")
print(f"Test Loss : {test_loss:.4f}")
print(f"WT Dice   : {wt_mean:.4f}")
print(f"TC Dice   : {tc_mean:.4f}")
print(f"ET Dice   : {et_mean:.4f}")
print(f"Mean Dice : {mean_dice:.4f}")

Test:   0%|          | 0/14 [00:00<?, ?it/s]


FINAL TEST RESULTS
Test Loss : 0.4109
WT Dice   : 0.8647
TC Dice   : 0.7844
ET Dice   : 0.7505
Mean Dice : 0.7999
